# Implementing the Edmonds-Karp Algorithm for Maximum Flow

This lecture covers the detailed implementation of the **Edmonds-Karp algorithm**, an application of the Ford-Fulkerson method that uses **Breadth-First Search (BFS)** to find shortest augmenting paths in a flow network.

---

## 1. Class Structure and Data Representation
The implementation is encapsulated within a `MaxFlow` class. It uses specific data structures to manage the graph and flow information efficiently.

### Private Entities
* **V**: An integer representing the number of vertices.
* **Edge List (EL)**: Stores triples consisting of the target vertex, the capacity, and the current flow for each edge.
* **Adjacency List (AL)**: Instead of storing neighbor IDs directly, it stores **indices** pointing to entries in the Edge List.
* **Distance Array (D)**: Used during the BFS to track levels and visit status.
* **Parent Array (P)**: Stores pairs of integers `(parent_id, edge_index)`. This is critical for backtracking to find the bottleneck capacity and updating the flow along the path.


---

## 2. Adding Edges to the Network
The `addEdge` function handles the creation of both forward and backward edges. By default, graphs are treated as directed.

### Logic for Edge Creation
1.  **Self-Loop Check**: The function ignores any edge where the source and destination are the same.
2.  **Forward Edge**: Added to the Edge List with the given capacity and an initial flow of 0. The index of this edge is added to the source's adjacency list.
3.  **Reverse (Back) Edge**: 
    * For **directed** graphs: Added with a capacity of 0.
    * For **undirected** graphs: Added with the same capacity as the forward edge.
4.  **Indexing Trick**: Edges are added in pairs (forward then backward). This allows the reverse edge of any edge at index `i` to be found using the bitwise operation `i XOR 1`.

---

## 3. Finding Paths with Breadth-First Search (BFS)
The BFS ensures we find the **shortest** augmenting path (in terms of the number of edges) in the residual graph.

### BFS Rules
* An edge is only traversable if its **residual capacity** (Capacity - Flow) is strictly greater than 0.
* Instead of deleting edges with zero residual capacity, they are simply skipped during the search.
* The search terminates early if it reaches the target vertex `T`.
* During traversal, the `Parent Array` records which vertex and which specific edge index lead to the current node.

---

## 4. Sending Flow through Augmenting Paths
Once a path is found, the `sendOneFlow` function (implemented recursively) identifies the bottleneck capacity and updates the flow values.

### Bottleneck Discovery and Flow Update
The function climbs back from the target `T` to the source `S` using the parent pointers.

```python
# Pseudocode for recursive flow update
Function send_one_flow(source, current_node, current_bottleneck):
    If source == current_node:
        Return current_bottleneck
    
    # Identify parent and edge index from BFS results
    parent, edge_idx = parent_info[current_node]
    
    # Calculate residual capacity of this edge
    res_cap = edge_list[edge_idx].capacity - edge_list[edge_idx].flow
    
    # Update bottleneck for the recursive call
    new_bottleneck = min(current_bottleneck, res_cap)
    
    # Recursive call to find the bottleneck of the entire path
    pushed_amount = send_one_flow(source, parent, new_bottleneck)
    
    # Update flows along the path
    edge_list[edge_idx].flow += pushed_amount
    edge_list[edge_idx XOR 1].flow -= pushed_amount
    
    Return pushed_amount
```

---

## 5. The Edmonds-Karp Algorithm
The main algorithm ties the components together in a loop that continues as long as an augmenting path exists.

### Main Execution Logic
1.  Initialize `total_max_flow` to 0.
2.  **While** `BFS(source, target)` finds a path:
    * Call `sendOneFlow` to determine the bottleneck capacity `f` and update edge flows.
    * Add `f` to `total_max_flow`.
3.  Return `total_max_flow`.

```python
# Pseudocode for the main Edmonds-Karp loop
Function calculate_max_flow(s, t):
    max_flow = 0
    While BFS(s, t) is True:
        pushed = send_one_flow(s, t, infinity)
        max_flow = max_flow + pushed
    Return max_flow
```

---

## Summary of Main Takeaways
* **Edmonds-Karp** is a specific implementation of Ford-Fulkerson using **BFS**, ensuring a polynomial time complexity.
* **Residual Graphs** are handled implicitly by checking the difference between capacity and flow during BFS.
* The **XOR indexing trick** (`i ^ 1`) is an efficient way to manage and update forward and backward edge pairs in a shared edge list.
* The algorithm terminates when no path with positive residual capacity exists from the source to the sink.